# Carga da série histórica nacional (NEX-GDDP-CMIP6)

Parte 1 — setup: imports e conexões (S3 + Postgres).


In [1]:
import os 
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import xarray as xr
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from botocore.config import Config
import sys
sys.path.append("../scripts") # achar o .py

from recortar_brasil import recortar_brasil

In [2]:
load_dotenv()

BUCKET = "nex-gddp-cmip6"
BASE_PREFIX = "NEX-GDDP-CMIP6/"

client = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED, connect_timeout=10, read_timeout=30, retries={"max_attempts": 2})
)
usuario = os.environ["DB_USER"]
senha = os.environ["DB_PASSWORD"]
host = os.environ.get("DB_HOST", "localhost")
porta = os.environ.get("DB_PORT", "5432")
nome_banco = os.environ["DB_NAME"]
engine = create_engine(f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{nome_banco}")

destino_tmp = "data/raw/_tmp_ano.nc"

In [3]:
# teste de conexão
engine.connect()
print("conectado")

conectado


#### Funções de baixar ano e agregar para série nacional


In [1]:

def baixar_ano(client, model, scenario, variable, ano, destino_tmp):
  for sufixo in [""]:
    print("ok")
    key = f"{BASE_PREFIX}{model}/{scenario}/r1i1p1f1/{variable}/{variable}_day_{model}_{scenario}_r1i1p1f1_gn_{ano}{sufixo}.nc"
    try:
      client.download_file(BUCKET, key, destino_tmp)
      return key
    except Exception:
      continue
  raise FileNotFoundError(f"Nenhuma verão encontrada para {variable}/{ano}")

In [2]:
def agregar_nacional(destino_tmp, variable):
    ds = xr.open_dataset(destino_tmp)
    ds_brasil = recortar_brasil(ds)
    da = ds_brasil[variable]
    media_diaria = da.mean(dim=["lat", "lon"], skipna=True)
    df = media_diaria.to_dataframe().reset_index()
    df = df.rename(columns={variable: "valor"})
    df["variavel"] = variable
    return df[["time", "variavel", "valor"]]